# Metabolite Mapper
Map the metabolites in the study with available name mappings under datasets\naming

In [1]:
""" Python script that applies syonym-mapping-new.json to studies downloaded
    studies from Metabolitics Workbench
"""
import os

import pandas as pd
from sklearn_utils.preprocessing import FeatureRenaming
from sklearn_utils.utils import SkUtilsIO

# Metabolitics pipeline step to use name mapping
from preprocessing.metabolitics_pipeline import MetaboliticsPipeline
from utils import load_metabolite_mapping
from utils.constants import (
    DATASET_ROOT_PATH,
    MAPPED_WORKBENCH_PATH,
    UNMAPPED_WORKBENCH_PATH,
)


def apply_mapping(x_data, y_data):
    """Apply mapping to unmapped labelled Metabolitics Workbench .csv file
    and return mapped version of it in a .csv format.
    """
    # Choose mapping json
    MetaboliticsPipeline.steps["metabolite-name-mapping"] = FeatureRenaming(
        load_metabolite_mapping("new-synonym")
    )

    # Apply pipeline step
    transformer_pipe = MetaboliticsPipeline(["metabolite-name-mapping"])

    X_transformed = transformer_pipe.fit_transform(X=x_data, y=y_data)

    X_df = pd.DataFrame(X_transformed)
    y_df = pd.DataFrame({"Factors": y_data})
    mapped_df = pd.concat([y_df, X_df], axis=1)
    # print(mapped_df.head(10))

    return mapped_df


### Example Study

In [ ]:
X_study, y_study = SkUtilsIO(f"{UNMAPPED_WORKBENCH_PATH}\\ST000002.csv").from_csv(label_column='Factors')

mapped_df = apply_mapping(X_study, y_study)

csv_file_name = "ST000002.csv"
save_path = os.path.join(MAPPED_WORKBENCH_PATH, csv_file_name)

mapped_df.to_csv(save_path, index=False)

### Bulk mapping

In [3]:
for filename in os.listdir(UNMAPPED_WORKBENCH_PATH):
    if filename.endswith(".csv"):
        try:
            file_path = os.path.join(UNMAPPED_WORKBENCH_PATH, filename)
            # Load data
            X_study, y_study = SkUtilsIO(file_path).from_csv(label_column='Factors')
            # Apply mapping
            mapped_df = apply_mapping(X_study, y_study)
            # Save to a .csv file
            save_path = os.path.join(MAPPED_WORKBENCH_PATH, filename)
            mapped_df.to_csv(save_path, index=False)
        except TypeError as error:
            print(f"TypeError in file {filename}: {str(error)}")

TypeError in file ST001406.csv: must be real number, not str
TypeError in file ST001907.csv: must be real number, not str
